In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
# Данные соревнования доступны только для чтения в /kaggle/input

test = pd.read_csv("/kaggle/input/datasets/fadinaifeh/flightdelaysfall2018/flight_delays_test.csv/flight_delays_test.csv")
train = pd.read_csv("/kaggle/input/datasets/fadinaifeh/flightdelaysfall2018/flight_delays_train.csv/flight_delays_train.csv")
print('train:', train.shape, '| test:', test.shape)
train.head()


train: (100000, 9) | test: (100000, 8)


,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance,dep_delayed_15min
0,c-8,c-21,c-7,1934,AA,ATL,DFW,732,N
1,c-4,c-20,c-3,1548,US,PIT,MCO,834,N
2,c-9,c-2,c-5,1422,XE,RDU,CLE,416,N
3,c-11,c-25,c-6,1015,OO,DEN,MEM,872,N
4,c-10,c-7,c-6,1828,WN,MDW,OMA,423,Y


In [2]:
#ETL
def transform(df):
    df = df.copy()
    # Поля 'c-8', 'c-21', 'c-7' -> целые числа
    for col in ['Month', 'DayofMonth', 'DayOfWeek']:
        df[col] = df[col].str.replace('c-', '', regex=False).astype(int)
    #Час вылета из DepTime (hhmm)
    df['DepHour'] = (df['DepTime'] // 100).clip(0, 23).astype(int)
    df['IsWeekend'] = (df['DayOfWeek'] >= 6).astype(int)
    return df
    
train_fe = transform(train)
test_fe = transform(test)
# Целевая переменная 'Y'/'N' -> 1/0
train_fe['target'] = (train_fe['dep_delayed_15min'] == 'Y').astype(int)
# Сохранение обработанного датасета (load -> transform -> save)
os.makedirs('/kaggle/working', exist_ok=True)
train_fe.to_parquet('/kaggle/working/train_processed.parquet', index=False)
print('Сохранено:', train_fe.shape)
train_fe.head()

Сохранено: (100000, 12)


,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance,dep_delayed_15min,DepHour,IsWeekend,target
0,8,21,7,1934,AA,ATL,DFW,732,N,19,1,0
1,4,20,3,1548,US,PIT,MCO,834,N,15,0,0
2,9,2,5,1422,XE,RDU,CLE,416,N,14,0,0
3,11,25,6,1015,OO,DEN,MEM,872,N,10,1,0
4,10,7,6,1828,WN,MDW,OMA,423,Y,18,1,1
